# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: Ngo Dong At

I have used the following AI tools: Claude, ChatGPT, NotebookLM

I understand that my submission needs to be my own work: NDA

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [11]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [12]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [13]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [14]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [15]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('listen')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [16]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[2 1]'; Monster is at '[0 2]'
..M..
.....
.H...
.....
.....
Hunter chose action 'south'
Monster chose action 'stay'


Step 2: Hunter is at '[3 1]'; Monster is at '[0 2]'
..M..
.....
.....
.H...
.....
Hunter chose action 'east'
Monster chose action 'stay'


Step 3: Hunter is at '[3 2]'; Monster is at '[0 2]'
..M..
.....
.....
..H..
.....
Hunter chose action 'east'
Monster chose action 'stay'


Step 4: Hunter is at '[3 3]'; Monster is at '[0 2]'
..M..
.....
.....
...H.
.....
Hunter chose action 'north'
Monster chose action 'stay'


Step 5: Hunter is at '[2 3]'; Monster is at '[0 2]'
..M..
.....
...H.
.....
.....
Hunter chose action 'east'
Monster chose action 'south'


Step 6: Hunter is at '[2 4]'; Monster is at '[1 2]'
.....
..M..
....H
.....
.....
Hunter chose action 'east'
Monster chose action 'east'


Step 7: Hunter is at '[2 4]'; Monster is at '[1 3]'
.....
...M.
....H
.....
.....
Hunter chose action 'east'
Monster chose action 'east'


Step 8: Hunter is at '[2

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [17]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[8, 42, 10, nan, 77, nan, 25, 10, 53, 28, nan, nan, nan, nan, 8, nan, 12, 54, 76, 0, 66, nan, 60, nan, nan, 41, nan, 67, 94, nan, nan, 1, 59, 69, nan, 19, 69, nan, nan, nan, nan, 1, nan, 1, nan, 58, nan, 88, 6, nan, 74, nan, 30, nan, 70, 55, nan, nan, 62, nan, 32, 76, 14, nan, nan, 57, 6, nan, 27, 70, 6, 6, 76, nan, nan, nan, nan, 7, 31, 86, 0, nan, nan, nan, nan, 88, nan, 23, 57, 4, 82, nan, nan, 81, nan, 56, 19, nan, nan, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [18]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 45 times.
Average steps to catch the monster over 55 successful runs: 41.76


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [19]:
class ModelBasedHunterAgent:

    def __init__(self):
        self.last_known = None  # remembered [row, col] of the monster, or None

    def act(self, hunter_pos, monster_pos):
        # monster_pos is only not-None right after we chose \'listen\' last turn
        if monster_pos is not None:
            self.last_known = list(monster_pos)

        # no information yet (very first turn) -> listen to get a first fix
        if self.last_known is None:
            return "listen"

        dr = self.last_known[0] - hunter_pos[0]
        dc = self.last_known[1] - hunter_pos[1]

        if dr == 0 and dc == 0:
            # we walked all the way to where the monster used to be, but it has
            # since moved away -- re-listen to find it again
            return "listen"

        # move along whichever axis has the larger remaining distance
        if abs(dr) >= abs(dc):
            return "south" if dr > 0 else "north"
        else:
            return "east" if dc > 0 else "west"


model_based_hunter_agent_function = ModelBasedHunterAgent().act
model_based_hunter_agent_function([0, 0], None)

'listen'

In [20]:
np.random.seed(3)
hunter = ModelBasedHunterAgent()
arena_environment(hunter.act, monster_agent_function_simple,
                   n=8, max_steps=30, visualize=True, animation=False)

Step 1: Hunter is at '[1 3]'; Monster is at '[2 0]'
........
...H....
M.......
........
........
........
........
........
Hunter chose action 'listen'
Monster chose action 'west'


Step 2: Hunter is at '[1 3]'; Monster is at '[2 0]'
........
...H....
M.......
........
........
........
........
........
Hunter chose action 'west'
Monster chose action 'stay'


Step 3: Hunter is at '[1 2]'; Monster is at '[2 0]'
........
..H.....
M.......
........
........
........
........
........
Hunter chose action 'west'
Monster chose action 'stay'


Step 4: Hunter is at '[1 1]'; Monster is at '[2 0]'
........
.H......
M.......
........
........
........
........
........
Hunter chose action 'south'
Monster chose action 'stay'


Step 5: Hunter is at '[2 1]'; Monster is at '[2 0]'
........
........
MH......
........
........
........
........
........
Hunter chose action 'west'
Monster chose action 'east'


Step 6: Hunter is at '[2 0]'; Monster is at '[2 1]'
........
........
HM......
........
....

9

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [21]:
def run_hunter_experiment(n, max_steps, n_runs=100):
    steps = []
    for _ in range(n_runs):
        hunter = ModelBasedHunterAgent()
        result = arena_environment(hunter.act,
                                    monster_agent_function_simple,
                                    n=n, max_steps=max_steps,
                                    visualize=False, animation=False)
        steps.append(result)
    steps = np.array(steps, dtype=float)
    print(f"n={n:>2}x{n:<2} | failed {np.sum(np.isnan(steps)):3.0f}/{n_runs} times | "
          f"avg steps over {np.sum(~np.isnan(steps)):.0f} successful runs: {np.nanmean(steps):.2f}")
    return steps


np.random.seed(7)
steps_10 = run_hunter_experiment(n=10, max_steps=100, n_runs=100)
steps_30 = run_hunter_experiment(n=30, max_steps=200, n_runs=100)

n=10x10 | failed   0/100 times | avg steps over 100 successful runs: 10.09
n=30x30 | failed   0/100 times | avg steps over 100 successful runs: 25.48


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

> My hunter is a **model-based reflex agent**: it remembers the monster's last known
> position (its only piece of state) and always moves along whichever axis (row or
> column) currently has the larger distance to that remembered position - a simple
> greedy strategy that reduces the Manhattan distance to the target by exactly one
> square every single move. It only spends a turn on `listen` when it has no
> information yet, or when it has walked all the way to the monster's last known
> square only to find it empty (the monster wandered off in the meantime), so
> almost every turn is spent actually closing the distance rather than gathering
> information. This works well because the monster in this environment mostly
> `stay`s in place (probability 0.5) and only occasionally takes a small random
> step, so a `listen`-then-beeline strategy rarely goes stale before the hunter
> arrives.
>
> I do **not** use `teleport`. Teleporting moves the hunter to a *uniformly random*
> square, which on average is no closer to the monster than staying put - it
> throws away the useful information the hunter just gathered with `listen`. It
> would only make sense as a desperation move if the hunter got stuck (e.g., stuck
> oscillating near a wall), which does not happen with the axis-greedy strategy
> above.
>
> Arena size mostly affects the **number of moves needed after each `listen`**: the
> average Manhattan distance to a random target grows roughly linearly with `n`, so
> catching the monster in a 30x30 arena typically takes a few times longer than in a
> 10x10 arena (see the two experiments above), even though the *strategy* itself
> does not change. Because the hunter re-listens only when it truly needs to
> (arriving at a stale position), the failure rate stays low even at 30x30 as long
> as `max_steps` is generous enough for the larger distances involved.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

> Right now the monster agent function ignores the hunter's position completely
> (`monster_agent_function_simple` only looks at its own random-move probabilities),
> so it never actively avoids the hunter. A smarter monster agent could take
> `hunter_pos` into account and, for example, move away from the hunter whenever
> the hunter is within some detection radius (increasing the Manhattan distance
> instead of moving randomly), while still occasionally `stay`ing or moving randomly
> the rest of the time so it stays somewhat unpredictable and doesn't always retreat
> into a corner. This would directly hurt the greedy hunter strategy above, since
> the "last known position" would go stale faster and the hunter would need to
> `listen` more often (spending more energy on information-gathering relative to
> actually closing the distance).

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.